In [ ]:
import torch
import torch.nn.functional as F

from cfg_tools import load_config_files

from numb_project.constants import BASE, CONFIG_FOLDER, DEVICE
from numb_project.domain_module import get_domains_config
from numb_project.gw_model import MyGlobalWorkspace, get_global_workspace_mods
from numb_project.data_module import MnistDataModule

config = load_config_files(
    f"{CONFIG_FOLDER}",
    use_cli=False,
    load_files=["config.yaml"])[0]

domain_configs = get_domains_config(['image', 'digit'])

gw_mod, selection_mod, operation_mod, attention_mod, loss_mod = get_global_workspace_mods(config, domain_configs)

gw = MyGlobalWorkspace.load_from_checkpoint(
    "/home/lucasc/Projects/gwnumb/checkpoints/numb/test.ckpt",
    gw_mod=gw_mod,
    selection_mod=selection_mod,
    loss_mod=loss_mod,
    weights_only=False
)
gw.eval()
gw.to(DEVICE)

data_module = MnistDataModule(50)


def make_chain_task(
    digit_one_hot: torch.Tensor, base: int = BASE, start_chain=0, end_chain=10
) -> tuple[torch.Tensor, torch.Tensor]:
    batch_size = digit_one_hot.shape[0]
    device = digit_one_hot.device

    digit_idx = digit_one_hot.argmax(dim=1)
    right_addend_value = torch.randint(start_chain, end_chain, (batch_size,), device=device)

    target_idx = (digit_idx + right_addend_value) % base

    right_addend_onehot = F.one_hot(right_addend_value, num_classes=base).float()
    target_one_hot = F.one_hot(target_idx, num_classes=base).float()

    return right_addend_onehot, target_one_hot


def get_image_digit_batch(item):
    """CombinedLoader yields (batch_dict, batch_idx, dataloader_idx)."""
    batch_dict = item[0] if isinstance(item, tuple) else item
    return batch_dict[frozenset({"image", "digit"})]


@torch.no_grad()
def evaluate_addition_accuracy(
    model: MyGlobalWorkspace,
    dataloader,
    base: int = BASE,
    chain_length: int = BASE * 2,
    start_chain: int = 0,
    end_chain: int = 10,
    n_batches: int | None = None,
    device: str = DEVICE,
):
    model.eval()
    correct_at_last_step = 0
    total = 0
    step_correct = torch.zeros(chain_length, device=device)
    step_total = torch.zeros(chain_length, device=device)

    for batch_idx, item in enumerate(dataloader):
        if n_batches is not None and batch_idx >= n_batches:
            break

        group = get_image_digit_batch(item)
        image = group["image"].to(device)
        left_digit_one_hot = group["digit"].to(device)

        right_addend_onehot, target_one_hot = make_chain_task(
            left_digit_one_hot, base=base, start_chain=start_chain, end_chain=end_chain
        )
        target_idx = target_one_hot.argmax(dim=1)

        cumulative_preds = model.forward_chain(
            image, right_addend_onehot, left_digit_one_hot, chain_length=chain_length
        )
        batch_size = image.shape[0]

        # accuracy au dernier pas de la chaîne
        preds_at_last_step = cumulative_preds[:, -1, :]
        pred_idx = preds_at_last_step.argmax(dim=1)
        correct_at_last_step += (pred_idx == target_idx).sum().item()
        total += batch_size

        # accuracy pas par pas contre (left_digit + t) % base
        left_idx = left_digit_one_hot.argmax(dim=1)
        for t in range(chain_length):
            expected_t = (left_idx + t) % base
            pred_t = cumulative_preds[:, t, :].argmax(dim=1)
            step_correct[t] += (pred_t == expected_t).sum()
            step_total[t] += batch_size

    last_step_acc = correct_at_last_step / total
    per_step_acc = (step_correct / step_total).cpu()

    print(f"Accuracy au dernier pas ({chain_length - 1}) : {last_step_acc:.4f} ({correct_at_last_step}/{total})")
    print("\nAccuracy par pas t :")
    for t, acc in enumerate(per_step_acc):
        print(f"  t={t:2d} : {acc:.4f}")

    return last_step_acc, per_step_acc

@torch.no_grad()
def show_addition_examples(
    model: MyGlobalWorkspace,
    dataloader,
    base: int = BASE,
    chain_length: int = BASE * 2,
    n_examples: int = 5,
    device: str = DEVICE,
    end_chain=3
):
    model.eval()
    item = next(iter(dataloader))
    group = get_image_digit_batch(item)

    image = group["image"][:n_examples].to(device)
    left_digit_one_hot = group["digit"][:n_examples].to(device)

    right_addend_onehot, target_one_hot = make_chain_task(left_digit_one_hot, base=base, end_chain=end_chain)
    target_idx = target_one_hot.argmax(dim=1)
    right_addend_value = right_addend_onehot.argmax(dim=1)

    cumulative_preds = model.forward_chain(
        image, right_addend_onehot, left_digit_one_hot, chain_length=chain_length
    )
    left_idx = left_digit_one_hot.argmax(dim=1)

    for i in range(n_examples):
        steps_needed = right_addend_value[i].item() + 1
        pred_seq = cumulative_preds[i, :, :].argmax(dim=1).cpu().tolist()
        pred_at_target = pred_seq[steps_needed] if steps_needed < chain_length else None
        status = "OK" if pred_at_target == target_idx[i].item() else "FAIL"
        print(
            f"Exemple {i}: {left_idx[i].item()} + {right_addend_value} = {target_idx[i].item()} "
            f"(mod {base}) | prédiction au pas {steps_needed} = {pred_at_target} [{status}]"
        )
        print(f"  séquence complète (t=0..{chain_length-1}): {pred_seq}")


# --- usage ---
test_loader = data_module.test_dataloader()

evaluate_addition_accuracy(gw, test_loader, chain_length=6, n_batches=20, end_chain=3)
show_addition_examples(gw, test_loader, chain_length=6, n_examples=5, end_chain=3)

Accuracy au dernier pas (5) : 0.9440 (944/1000)

Accuracy par pas t :
  t= 0 : 0.3420
  t= 1 : 0.3120
  t= 2 : 0.2970
  t= 3 : 0.0050
  t= 4 : 0.0050
  t= 5 : 0.0100
Exemple 0: 8 + tensor([0, 0, 0, 1, 0], device='cuda:0') = 8 (mod 10) | prédiction au pas 1 = 8 [OK]
  séquence complète (t=0..5): [8, 8, 8, 8, 8, 8]
Exemple 1: 0 + tensor([0, 0, 0, 1, 0], device='cuda:0') = 0 (mod 10) | prédiction au pas 1 = 0 [OK]
  séquence complète (t=0..5): [0, 0, 0, 0, 0, 0]
Exemple 2: 1 + tensor([0, 0, 0, 1, 0], device='cuda:0') = 1 (mod 10) | prédiction au pas 1 = 1 [OK]
  séquence complète (t=0..5): [1, 1, 1, 1, 1, 1]
Exemple 3: 3 + tensor([0, 0, 0, 1, 0], device='cuda:0') = 4 (mod 10) | prédiction au pas 2 = 4 [OK]
  séquence complète (t=0..5): [4, 4, 4, 4, 4, 4]
Exemple 4: 5 + tensor([0, 0, 0, 1, 0], device='cuda:0') = 5 (mod 10) | prédiction au pas 1 = 5 [OK]
  séquence complète (t=0..5): [5, 5, 5, 5, 5, 5]
